# 01 - Shot detection and keyframe extraction (all videos)

Quét toàn bộ MP4 trong `/kaggle/input`, xử lý tuần tự, checkpoint theo video và tự bỏ qua video đã hoàn thành.


In [ ]:
%pip install -q ffmpeg-python av open_clip_torch scikit-learn pyarrow
!command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -qq ffmpeg)
!test -d /kaggle/working/TransNetV2 || git clone -q --depth 1 https://github.com/soCzech/TransNetV2.git /kaggle/working/TransNetV2

In [ ]:
from __future__ import annotations

import json
import math
import shutil
import sys
from pathlib import Path

import av
import cv2
import numpy as np
import open_clip
import pandas as pd
import torch
from PIL import Image
from sklearn.cluster import AgglomerativeClustering

PIPELINE_VERSION = 'aicv3-keyframes'
BATCH_NOTEBOOK_VERSION = '2026-08-18-no-btc-checkpoints'
SOURCE_KIND = 'self_extracted'
MODEL_NAME = 'ViT-B-32'
MODEL_PRETRAINED = 'laion2b_s34b_b79k'
MIN_SHOT_SECONDS = 0.6
CANDIDATE_STRIDE_SECONDS = 1.0
LONG_WINDOW_SECONDS = 5.0
MAX_CANDIDATES_PER_SHOT = 32
MAX_KEYFRAMES_PER_SHOT = 4
DUPLICATE_HASH_DISTANCE = 6
# BTC reference checkpoints belong to the single-video validation notebook only.
# Disable them in batch because each video has different FPS and frame count.
BTC_FRAME_CHECKPOINTS = {}
FRAME_TIME_TOLERANCE_SEC = 1.0 / 30.0 + 0.02
CLUSTER_DISTANCE_THRESHOLD = 0.22
CLIP_BATCH_SIZE = 64

INPUT_ROOT = Path('/kaggle/input')
BATCH_ROOT = Path('/kaggle/working/01-shot-keyframe-all-videos-output')
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
VIDEO_PATHS = sorted(INPUT_ROOT.rglob('*.mp4'))
if not VIDEO_PATHS:
    raise FileNotFoundError('Không tìm thấy file MP4 trong /kaggle/input')
print('Batch notebook version:', BATCH_NOTEBOOK_VERSION)
print('BTC checkpoints:', BTC_FRAME_CHECKPOINTS)
print('Videos found:', len(VIDEO_PATHS))

if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật GPU trong Kaggle Notebook Settings')

print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
TRANSNET_INFERENCE = Path('/kaggle/working/TransNetV2/inference')
sys.path.insert(0, str(TRANSNET_INFERENCE))
from transnetv2 import TransNetV2

device = torch.device('cuda')
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=MODEL_PRETRAINED
)
clip_model = clip_model.to(device).eval()
transnet_model = TransNetV2()
print('Models ready')

In [ ]:
def perceptual_hash(frame: np.ndarray) -> int:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA).astype('float32')
    dct = cv2.dct(resized)[:8, :8]
    values = dct.flatten()[1:]
    median = float(np.median(values))
    return sum((int(value > median) << index) for index, value in enumerate(values))


def hamming_distance(left: int, right: int) -> int:
    return (left ^ right).bit_count()


def score_frame(frame: np.ndarray) -> tuple[float, dict]:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    mean_luma = float(gray.mean())
    black_ratio = float((gray < 16).mean())
    white_ratio = float((gray > 245).mean())
    exposure = max(0.0, 1.0 - abs(mean_luma - 127.5) / 127.5)
    sharpness_score = min(1.0, math.log1p(sharpness) / math.log1p(1500.0))
    score = 0.60 * sharpness_score + 0.25 * exposure + 0.15 * (1.0 - max(black_ratio, white_ratio))
    return score, {
        'sharpness': sharpness, 'mean_luma': mean_luma,
        'black_ratio': black_ratio, 'white_ratio': white_ratio,
        'phash': perceptual_hash(frame),
    }


def encode_frames(frames: list[np.ndarray]) -> np.ndarray:
    vectors = []
    for start in range(0, len(frames), CLIP_BATCH_SIZE):
        tensors = [
            clip_preprocess(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            for frame in frames[start:start + CLIP_BATCH_SIZE]
        ]
        batch = torch.stack(tensors).to(device)
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            features = clip_model.encode_image(batch)
            features = torch.nn.functional.normalize(features.float(), dim=-1)
        vectors.append(features.cpu().numpy().astype('float32'))
    return np.concatenate(vectors, axis=0)


def cluster_representatives(features: np.ndarray, qualities: np.ndarray) -> list[int]:
    if len(features) == 1:
        return [0]
    labels = AgglomerativeClustering(
        n_clusters=None, metric='cosine', linkage='average',
        distance_threshold=CLUSTER_DISTANCE_THRESHOLD,
    ).fit_predict(features)
    selected = []
    for label in np.unique(labels):
        group = np.flatnonzero(labels == label)
        centroid = features[group].mean(axis=0)
        centroid /= max(np.linalg.norm(centroid), 1e-12)
        scores = 0.80 * (features[group] @ centroid) + 0.20 * qualities[group]
        selected.append(int(group[int(np.argmax(scores))]))
    return selected


def dynamic_keyframe_limit(duration_sec: float) -> int:
    if duration_sec <= 3.0:
        return 1
    if duration_sec <= 10.0:
        return 2
    if duration_sec <= 25.0:
        return 3
    return MAX_KEYFRAMES_PER_SHOT


In [ ]:
import gc
import traceback

success_count = 0
failure_count = 0
failures = []

for video_number, VIDEO_PATH in enumerate(VIDEO_PATHS, start=1):
    VIDEO_ID = VIDEO_PATH.stem
    RUN_ID = f'{PIPELINE_VERSION}-{VIDEO_ID.lower()}'
    OUTPUT_ROOT = BATCH_ROOT / VIDEO_ID
    FRAME_ROOT = OUTPUT_ROOT / 'frames' / VIDEO_ID
    SUCCESS_PATH = OUTPUT_ROOT / '_SUCCESS.json'
    ERROR_PATH = OUTPUT_ROOT / '_ERROR.txt'

    if SUCCESS_PATH.exists():
        print(f'[{video_number}/{len(VIDEO_PATHS)}] Skip completed {VIDEO_ID}')
        success_count += 1
        continue

    print(f'\n[{video_number}/{len(VIDEO_PATHS)}] Processing {VIDEO_ID}: {VIDEO_PATH}')
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    FRAME_ROOT.mkdir(parents=True, exist_ok=True)

    try:
        cap = cv2.VideoCapture(str(VIDEO_PATH))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        if fps <= 0 or frame_count <= 0:
            raise RuntimeError('Không đọc được FPS/frame_count của video')

        _, single_predictions, _ = transnet_model.predict_video(str(VIDEO_PATH))
        raw_scenes = transnet_model.predictions_to_scenes(single_predictions)
        raw_scene_ranges = [(int(start), min(frame_count, int(end) + 1)) for start, end in raw_scenes]
        min_shot_frames = max(1, round(MIN_SHOT_SECONDS * fps))
        scenes = []
        for start_frame, end_frame in raw_scene_ranges:
            if scenes and end_frame - start_frame < min_shot_frames:
                scenes[-1] = (scenes[-1][0], end_frame)
            else:
                scenes.append((start_frame, end_frame))
        if not scenes:
            scenes = [(0, frame_count)]

        print(f'fps={fps:.3f} frames={frame_count} resolution={width}x{height}')
        print('Detected shots:', len(scenes))

        # Build all candidate source-frame indices before decoding.
        shot_rows = []
        shot_plans = []
        for shot_index, (start_frame, end_frame) in enumerate(scenes):
            shot_id = f'{VIDEO_ID}:self:{PIPELINE_VERSION}:shot:{shot_index:06d}'
            duration = (end_frame - start_frame) / fps
            shot_rows.append({
                'shot_id': shot_id, 'video_id': VIDEO_ID, 'shot_index': shot_index,
                'start_frame': start_frame, 'end_frame': end_frame,
                'start_time': start_frame / fps, 'end_time': end_frame / fps,
                'duration_sec': duration, 'detector': 'transnetv2',
            })
            candidate_count = min(128, max(1, math.ceil(duration / CANDIDATE_STRIDE_SECONDS)))
            indices = np.unique(np.linspace(start_frame, max(start_frame, end_frame - 1), candidate_count, dtype=int))
            shot_plans.append({
                'shot_id': shot_id, 'duration': duration, 'start_frame': start_frame,
                'indices': set(int(value) for value in indices),
            })

        wanted_frames = {}
        for shot_index, plan in enumerate(shot_plans):
            for frame_idx in plan['indices']:
                window_id = int((frame_idx - plan['start_frame']) / max(1, fps * LONG_WINDOW_SECONDS))
                wanted_frames.setdefault(frame_idx, []).append((shot_index, window_id))

        accepted_by_shot = [[] for _ in shot_plans]
        accepted_hashes = [[] for _ in shot_plans]
        best_raw_by_shot = [None for _ in shot_plans]
        decoded_checkpoints = {}
        last_decoded_frame_idx = -1

        def compress_candidate(frame_bgr: np.ndarray) -> np.ndarray:
            ok, encoded = cv2.imencode('.jpg', frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
            if not ok:
                raise RuntimeError('Không nén được candidate frame')
            return encoded

        container = av.open(str(VIDEO_PATH))
        stream = container.streams.video[0]
        for decoded_frame_idx, decoded_frame in enumerate(container.decode(stream)):
            last_decoded_frame_idx = decoded_frame_idx
            pts = decoded_frame.pts
            time_base = decoded_frame.time_base
            pts_time = float(pts * time_base) if pts is not None and time_base is not None else decoded_frame_idx / fps
            if decoded_frame_idx in BTC_FRAME_CHECKPOINTS:
                decoded_checkpoints[decoded_frame_idx] = pts_time
            targets = wanted_frames.get(decoded_frame_idx)
            if not targets:
                continue
            frame_bgr = decoded_frame.to_ndarray(format='bgr24')
            quality, metrics = score_frame(frame_bgr)
            encoded = compress_candidate(frame_bgr)
            for shot_index, window_id in targets:
                record = {
                    'frame_idx': decoded_frame_idx, 'pts': pts, 'pts_time': pts_time,
                    'time_base': str(time_base), 'encoded': encoded, 'quality': quality,
                    'metrics': metrics, 'window_id': window_id, 'quality_fallback': True,
                }
                current_best = best_raw_by_shot[shot_index]
                if current_best is None or quality > current_best['quality']:
                    best_raw_by_shot[shot_index] = record
                if metrics['black_ratio'] > 0.60 or metrics['white_ratio'] > 0.60:
                    continue
                if metrics['sharpness'] < 25.0:
                    continue
                if any(hamming_distance(metrics['phash'], value) < DUPLICATE_HASH_DISTANCE for value in accepted_hashes[shot_index]):
                    continue
                accepted_hashes[shot_index].append(metrics['phash'])
                accepted_by_shot[shot_index].append({**record, 'quality_fallback': False})
        container.close()

        if last_decoded_frame_idx + 1 != frame_count:
            raise RuntimeError(
                f'PyAV decoded {last_decoded_frame_idx + 1} frames nhưng ffprobe/OpenCV báo {frame_count}'
            )

        frame_mapping_report = []
        for expected_idx, expected_time in BTC_FRAME_CHECKPOINTS.items():
            actual_time = decoded_checkpoints.get(expected_idx)
            if actual_time is None:
                raise RuntimeError(f'Không decode được checkpoint BTC frame {expected_idx}')
            error_sec = abs(actual_time - expected_time)
            frame_mapping_report.append({
                'frame_idx': expected_idx, 'btc_pts_time': expected_time,
                'decoded_pts_time': actual_time, 'error_sec': error_sec,
                'passed': error_sec <= FRAME_TIME_TOLERANCE_SEC,
            })
        if not all(item['passed'] for item in frame_mapping_report):
            raise RuntimeError(f'Frame mapping không khớp BTC: {frame_mapping_report}')
        (OUTPUT_ROOT / 'frame-mapping-report.json').write_text(
            json.dumps(frame_mapping_report, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        print('BTC frame mapping:', frame_mapping_report)

        keyframe_rows, selected_vectors = [], []
        ordinal = 0
        for shot_index, plan in enumerate(shot_plans):
            candidates = accepted_by_shot[shot_index]
            if not candidates and best_raw_by_shot[shot_index] is not None:
                candidates = [best_raw_by_shot[shot_index]]
            if not candidates:
                raise RuntimeError(f'Không decode được candidate cho {plan["shot_id"]}')
            frames = [cv2.imdecode(item['encoded'], cv2.IMREAD_COLOR) for item in candidates]
            if any(frame is None for frame in frames):
                raise RuntimeError(f'Candidate JPEG bị lỗi trong {plan["shot_id"]}')
            features = encode_frames(frames)
            selected_global = []
            for window_id in sorted({item['window_id'] for item in candidates}):
                local = np.asarray([i for i, item in enumerate(candidates) if item['window_id'] == window_id], dtype=int)
                local_features = features[local]
                local_qualities = np.asarray([candidates[i]['quality'] for i in local], dtype='float32')
                selected_global.extend(local[int(i)] for i in cluster_representatives(local_features, local_qualities))
            limit = dynamic_keyframe_limit(plan['duration'])
            selected_global = sorted(set(selected_global), key=lambda index: candidates[index]['frame_idx'])
            if len(selected_global) > limit:
                groups = np.array_split(np.asarray(selected_global), limit)
                selected_global = [int(max(group, key=lambda index: candidates[index]['quality'])) for group in groups if len(group)]
            for candidate_index in selected_global:
                item = candidates[candidate_index]
                frame_bgr = frames[candidate_index]
                ordinal += 1
                keyframe_id = f'{VIDEO_ID}:self:{PIPELINE_VERSION}:kf:{ordinal:06d}'
                frame_name = f'{VIDEO_ID}_self_{PIPELINE_VERSION}_kf_{ordinal:06d}.webp'
                frame_path = FRAME_ROOT / frame_name
                if not cv2.imwrite(str(frame_path), frame_bgr, [cv2.IMWRITE_WEBP_QUALITY, 92]):
                    raise RuntimeError(f'Không ghi được {frame_path}')
                metrics = item['metrics']
                keyframe_rows.append({
                    'keyframe_id': keyframe_id, 'video_id': VIDEO_ID, 'shot_id': plan['shot_id'],
                    'ordinal': ordinal, 'frame_idx': item['frame_idx'], 'pts': item['pts'],
                    'pts_time': item['pts_time'], 'time_base': item['time_base'], 'fps': fps,
                    'quality_score': item['quality'], 'sharpness': metrics['sharpness'],
                    'mean_luma': metrics['mean_luma'], 'black_ratio': metrics['black_ratio'],
                    'white_ratio': metrics['white_ratio'], 'window_id': item['window_id'],
                    'quality_fallback': item['quality_fallback'],
                    'frame_path': f'frames/{VIDEO_ID}/{frame_name}',
                    'source': SOURCE_KIND, 'pipeline_version': PIPELINE_VERSION, 'run_id': RUN_ID,
                    'extractor_version': 'transnetv2-pyav-phash-openclip-agglomerative-v3',
                })
                selected_vectors.append(features[candidate_index])
            if (shot_index + 1) % 20 == 0 or shot_index + 1 == len(shot_plans):
                print(f'{shot_index + 1}/{len(shot_plans)} shots, {ordinal} keyframes')

        if not keyframe_rows:
            raise RuntimeError('Không tạo được keyframe nào')


        shots_df = pd.DataFrame(shot_rows)
        keyframes_df = pd.DataFrame(keyframe_rows)
        selection_vectors = np.asarray(selected_vectors, dtype="float32")

        # Quality gates before writing artifacts.
        assert len(keyframes_df) == len(selection_vectors) > 0
        assert keyframes_df["keyframe_id"].is_unique
        assert set(shots_df["shot_id"]) == set(keyframes_df["shot_id"]), "Mỗi shot phải có ít nhất một keyframe"
        assert keyframes_df["frame_idx"].between(0, frame_count - 1).all()
        assert keyframes_df["pts_time"].between(0, frame_count / fps).all()
        assert np.isfinite(selection_vectors).all()
        selection_norms = np.linalg.norm(selection_vectors, axis=1)
        assert np.allclose(selection_norms, 1.0, atol=1e-4)

        shot_durations = shots_df["duration_sec"]
        coverage_start = float(keyframes_df["pts_time"].min())
        coverage_end = float(keyframes_df["pts_time"].max())
        coverage_ratio = (coverage_end - coverage_start) / (frame_count / fps)
        quality_report = {
            "unique_keyframe_ids": bool(keyframes_df["keyframe_id"].is_unique),
            "shot_duration_min_sec": float(shot_durations.min()),
            "shot_duration_median_sec": float(shot_durations.median()),
            "shot_duration_max_sec": float(shot_durations.max()),
            "keyframes_per_minute": float(len(keyframes_df) / (frame_count / fps / 60)),
            "temporal_coverage_ratio": float(coverage_ratio),
            "quality_score_min": float(keyframes_df["quality_score"].min()),
            "quality_score_median": float(keyframes_df["quality_score"].median()),
            "quality_score_max": float(keyframes_df["quality_score"].max()),
            "quality_fallback_count": int(keyframes_df["quality_fallback"].sum()),
            "shots_without_keyframe": int((~shots_df["shot_id"].isin(keyframes_df["shot_id"])).sum()),
            "passed": bool(coverage_ratio >= 0.90),
        }

        shots_df.to_parquet(OUTPUT_ROOT / "shots.parquet", index=False)
        keyframes_df.to_parquet(OUTPUT_ROOT / "keyframes.parquet", index=False)
        summary = {
            "stage": "shot-keyframe",
            "pipeline_version": PIPELINE_VERSION,
            "video_id": VIDEO_ID,
            "source": SOURCE_KIND,
            "run_id": RUN_ID,
            "video_path": str(VIDEO_PATH),
            "fps": fps,
            "frame_count": frame_count,
            "shots": len(shots_df),
            "keyframes": len(keyframes_df),
            "shot_detector": "TransNetV2",
            "keyframe_selector": f"quality+pHash+{MODEL_NAME}/{MODEL_PRETRAINED}+agglomerative",
            "candidate_stride_seconds": CANDIDATE_STRIDE_SECONDS,
            "long_window_seconds": LONG_WINDOW_SECONDS,
            "cluster_distance_threshold": CLUSTER_DISTANCE_THRESHOLD,
            "duplicate_hash_distance": DUPLICATE_HASH_DISTANCE,
            "max_keyframes_per_shot": MAX_KEYFRAMES_PER_SHOT,
            "frame_mapping_report": frame_mapping_report,
            "quality_report": quality_report,
        }
        (OUTPUT_ROOT / "summary.json").write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        # The selection vectors are temporary. Visual retrieval vectors belong to stage 02.
        import gc
        # Keep TransNetV2 and OpenCLIP loaded for the next video in the batch.
        for _name in ("selected_vectors", "selection_vectors"):
            if _name in globals():
                del globals()[_name]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        archive = shutil.make_archive(
            str(BATCH_ROOT / f"01-shot-keyframe-{VIDEO_ID}"), "zip", root_dir=OUTPUT_ROOT
        )
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        print("Download:", archive)
        SUCCESS_PATH.write_text(json.dumps({
            'video_id': VIDEO_ID,
            'video_path': str(VIDEO_PATH),
            'shots': int(len(shots_df)),
            'keyframes': int(len(keyframes_df)),
            'pipeline_version': PIPELINE_VERSION,
        }, ensure_ascii=False, indent=2), encoding='utf-8')
        ERROR_PATH.unlink(missing_ok=True)
        success_count += 1
        print(f'Completed {VIDEO_ID}: shots={len(shots_df)} keyframes={len(keyframes_df)}')
    except Exception as exc:
        failure_count += 1
        error_text = traceback.format_exc()
        ERROR_PATH.write_text(error_text, encoding='utf-8')
        failures.append({'video_id': VIDEO_ID, 'error': f'{type(exc).__name__}: {exc}'})
        print(f'ERROR {VIDEO_ID}: {type(exc).__name__}: {exc}')
    finally:
        for name in ('shots_df', 'keyframes_df', 'selection_vectors', 'selected_vectors', 'keyframe_rows', 'shot_rows'):
            globals().pop(name, None)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

batch_summary = {
    'videos_found': len(VIDEO_PATHS),
    'completed': success_count,
    'failed': failure_count,
    'failures': failures,
}
(BATCH_ROOT / 'batch-summary.json').write_text(
    json.dumps(batch_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(batch_summary, ensure_ascii=False, indent=2))
if failure_count:
    print('Có video lỗi. Xem _ERROR.txt trong thư mục của từng video; chạy lại notebook sẽ retry.')


## Output

`/kaggle/working/01-shot-keyframe-all-videos-output/<VIDEO_ID>/` chứa `shots.parquet`, `keyframes.parquet`, embedding lựa chọn và frames.
